# Bi-GRU for Protein Secondary Structure Prediction

This notebook implements a **Bidirectional GRU (Gated Recurrent Unit)** architecture for sequence-to-sequence prediction of protein secondary structures.

## Architecture Highlights
- **Stacked Bi-GRU**: 2-3 layers with 256-512 hidden units
- **Dropout Regularization**: 0.3 between layers
- **Multi-Task Learning**: Simultaneous Q8 and Q3 prediction
- **Model Checkpointing**: Save best models automatically
- **Target**: F1 Score ≥ 0.60

## Why GRU over LSTM?
Fewer parameters: ~25% fewer than LSTM (faster training)
Simpler architecture: 2 gates instead of 3
Often comparable performance: Similar accuracy to LSTM


## 1. Setup and Configuration

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json

# Configuration
class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    
    # Model architecture
    embed_dim = 128
    hidden_dim = 512  # Increased from 256 for better capacity
    num_layers = 3    # Stacked LSTM layers
    dropout = 0.3
    
    # Training
    batch_size = 64
    learning_rate = 0.001
    weight_decay = 1e-5
    epochs = 30
    grad_clip = 1.0
    
    # Data split
    val_split = 0.1
    
    # Paths
    train_path = "sep-25-dl-gen-ai-nppe-2/train.csv"
    test_path = "sep-25-dl-gen-ai-nppe-2/test.csv"
    model_save_path = "best_model_bigru.pth"
    submission_path = "submission_bigru.csv"
    
    # TrackIO
    trackio_project = "25-t3-nppe2"
    trackio_space_id = "manyakaistha/dlgenai-nppe"
    trackio_group = "bi_gru"

cfg = Config()
print(f"Using device: {cfg.device}")
print(f"Hidden dim: {cfg.hidden_dim}, Num layers: {cfg.num_layers}")

# Set random seeds
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

Using device: mps
Hidden dim: 512, Num layers: 3


/Volumes/1700 APFS/PERSONAL_PROGRAMMING_PROJECTS/intro_dl_nppe/nppe2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. TrackIO Setup

In [ ]:
# Optional: Set your HuggingFace token
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_token_here"
os.environ["TRACKIO_SPACE_ID"] = cfg.trackio_space_id

In [ ]:
def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN")
        if cfg.trackio_space_id and token:
            trackio.init(
                project=cfg.trackio_project,
                space_id=cfg.trackio_space_id,
                group=cfg.trackio_group
            )
            trackio.log({"status/started": 1})
            print("✓ TrackIO initialized")
            return trackio
    except Exception as e:
        print(f"TrackIO initialization failed: {e}")
    return None

trk = maybe_init_trackio()

## 3. Data Preparation

In [2]:
# Vocabulary mappings
PAD_TOKEN = 0

# Amino Acid vocabulary
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

# Q8 vocabulary (8 classes)
Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

# Q3 vocabulary (3 classes)
Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

print(f"Vocab Sizes: AA={len(AA_TO_ID)}, Q8={len(Q8_TO_ID)}, Q3={len(Q3_TO_ID)}")

Vocab Sizes: AA=23, Q8=9, Q3=4


In [3]:
# Dataset class
class ProteinDataset(Dataset):
    def __init__(self, data, mode='train'):
        self.data = data
        self.mode = mode
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_str = row['seq']
        
        # Encode sequence
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in seq_str]
        
        item = {
            'seq': torch.tensor(seq_ids, dtype=torch.long),
            'id': row['id']
        }
        
        if self.mode == 'train':
            q8_str = row['sst8']
            q3_str = row['sst3']
            
            q8_ids = [Q8_TO_ID[c] for c in q8_str]
            q3_ids = [Q3_TO_ID[c] for c in q3_str]
            
            item['q8'] = torch.tensor(q8_ids, dtype=torch.long)
            item['q3'] = torch.tensor(q3_ids, dtype=torch.long)
            
        return item

# Collate function
def collate_fn(batch):
    seqs = [item['seq'] for item in batch]
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN)
    
    ids = [item['id'] for item in batch]
    
    batch_out = {
        'seq': seqs_padded,
        'ids': ids,
        'lengths': torch.tensor([len(s) for s in seqs], dtype=torch.long)
    }
    
    if 'q8' in batch[0]:
        q8s = [item['q8'] for item in batch]
        q3s = [item['q3'] for item in batch]
        
        batch_out['q8'] = pad_sequence(q8s, batch_first=True, padding_value=PAD_TOKEN)
        batch_out['q3'] = pad_sequence(q3s, batch_first=True, padding_value=PAD_TOKEN)
        
    return batch_out

# Load and split data
df_train_full = pd.read_csv(cfg.train_path)
train_data, val_data = train_test_split(
    df_train_full, 
    test_size=cfg.val_split, 
    random_state=cfg.seed
)

train_ds = ProteinDataset(train_data.reset_index(drop=True), mode='train')
val_ds = ProteinDataset(val_data.reset_index(drop=True), mode='train')

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Batches per epoch: {len(train_loader)}")

Train samples: 6535
Validation samples: 727
Batches per epoch: 103


## 4. Bi-GRU Model Architecture

In [4]:
class BiGRU_MTL(nn.Module):
    """Bidirectional GRU (Gated Recurrent Unit) with Multi-Task Learning for Q8 and Q3 prediction."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, 
                 num_classes_q8, num_classes_q3, dropout=0.3):
        super(BiGRU_MTL, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size, 
            embedding_dim, 
            padding_idx=PAD_TOKEN
        )
        
        # Bi-GRU encoder (stacked)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0  # Dropout between GRU layers
        )
        
        # Dropout before classification heads
        self.dropout = nn.Dropout(dropout)
        
        # Classification heads
        # Input size is hidden_dim * 2 because of bidirectional
        self.q8_head = nn.Linear(hidden_dim * 2, num_classes_q8)
        self.q3_head = nn.Linear(hidden_dim * 2, num_classes_q3)
        
    def forward(self, x):
        # x: [batch, seq_len]
        embed = self.embedding(x)  # [batch, seq_len, embed_dim]
        
        # GRU output: [batch, seq_len, hidden_dim * 2]
        gru_out, _ = self.gru(embed)
        
        # Apply dropout
        gru_out = self.dropout(gru_out)
        
        # Classification heads
        q8_logits = self.q8_head(gru_out)  # [batch, seq_len, num_classes_q8]
        q3_logits = self.q3_head(gru_out)  # [batch, seq_len, num_classes_q3]
        
        return q8_logits, q3_logits

# Initialize model
model = BiGRU_MTL(
    vocab_size=len(AA_TO_ID),
    embedding_dim=cfg.embed_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_classes_q8=len(Q8_TO_ID),
    num_classes_q3=len(Q3_TO_ID),
    dropout=cfg.dropout
).to(cfg.device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

BiGRU_MTL(
  (embedding): Embedding(23, 128, padding_idx=0)
  (gru): GRU(128, 512, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (q8_head): Linear(in_features=1024, out_features=9, bias=True)
  (q3_head): Linear(in_features=1024, out_features=4, bias=True)
)

Total parameters: 11,437,965


## 5. Training Setup

In [7]:
# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

# Optimizer with weight decay
optimizer = optim.Adam(
    model.parameters(), 
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',  # Maximize F1 score
    factor=0.5,
    patience=3,
)

# Model checkpoint
class ModelCheckpoint:
    def __init__(self, filepath, monitor='val_f1_harmonic', mode='max'):
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best_score = -np.inf if mode == 'max' else np.inf
        
    def __call__(self, score, model, optimizer, epoch, metrics):
        improved = (score > self.best_score) if self.mode == 'max' else (score < self.best_score)
        
        if improved:
            print(f'✓ Saving best model (epoch {epoch+1}, {self.monitor}={score:.4f})')
            self.best_score = score
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_score': self.best_score,
                'metrics': metrics,
                'config': {
                    'embed_dim': cfg.embed_dim,
                    'hidden_dim': cfg.hidden_dim,
                    'num_layers': cfg.num_layers,
                    'dropout': cfg.dropout
                }
            }, self.filepath)
            return True
        return False

checkpoint = ModelCheckpoint(cfg.model_save_path)

print("Training setup complete!")

Training setup complete!


## 6. Metric Calculation

In [8]:
def calculate_f1(preds, targets, pad_idx=PAD_TOKEN):
    """Calculate macro F1 score, excluding padding tokens."""
    # Flatten
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    
    # Filter padding
    mask = targets_flat != pad_idx
    preds_masked = preds_flat[mask]
    targets_masked = targets_flat[mask]
    
    # Calculate F1
    return f1_score(targets_masked, preds_masked, average='macro', zero_division=0)

def harmonic_mean(a, b):
    """Calculate harmonic mean of two values."""
    if a + b == 0:
        return 0
    return 2 * (a * b) / (a + b)

## 7. Training and Validation Functions

In [9]:
def train_epoch(model, loader, optimizer, criterion, device, epoch):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_q8_f1 = 0
    total_q3_f1 = 0
    count = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} Train")
    for batch in pbar:
        seq = batch['seq'].to(device)
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        q8_logits, q3_logits = model(seq)
        
        # Calculate loss
        loss_q8 = criterion(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
        loss_q3 = criterion(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
        loss = loss_q8 + loss_q3
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        
        optimizer.step()
        
        # Calculate metrics
        total_loss += loss.item()
        f1_q8 = calculate_f1(q8_logits.detach(), q8_tgt)
        f1_q3 = calculate_f1(q3_logits.detach(), q3_tgt)
        total_q8_f1 += f1_q8
        total_q3_f1 += f1_q3
        count += 1
        
        # Update progress bar
        pbar.set_postfix({
            'loss': loss.item(), 
            'f1_q8': f1_q8, 
            'f1_q3': f1_q3
        })
        
        # Log to TrackIO (batch-level)
        if trk:
            trk.log({
                'train/loss': loss.item(),
                'train/loss_q8': loss_q8.item(),
                'train/loss_q3': loss_q3.item(),
                'train/f1_q8_batch': f1_q8,
                'train/f1_q3_batch': f1_q3
            })
    
    # Epoch averages
    avg_loss = total_loss / count
    avg_f1_q8 = total_q8_f1 / count
    avg_f1_q3 = total_q3_f1 / count
    avg_f1_harmonic = harmonic_mean(avg_f1_q8, avg_f1_q3)
    
    return avg_loss, avg_f1_q8, avg_f1_q3, avg_f1_harmonic

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    all_q8_preds = []
    all_q8_targets = []
    all_q3_preds = []
    all_q3_targets = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            seq = batch['seq'].to(device)
            q8_tgt = batch['q8'].to(device)
            q3_tgt = batch['q3'].to(device)
            
            # Forward pass
            q8_logits, q3_logits = model(seq)
            
            # Calculate loss
            loss_q8 = criterion(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            loss = loss_q8 + loss_q3
            
            total_loss += loss.item()
            
            # Collect predictions (excluding padding)
            q8_pred = q8_logits.argmax(dim=-1)
            q3_pred = q3_logits.argmax(dim=-1)
            
            mask = q8_tgt != PAD_TOKEN
            all_q8_preds.extend(q8_pred[mask].cpu().numpy())
            all_q8_targets.extend(q8_tgt[mask].cpu().numpy())
            all_q3_preds.extend(q3_pred[mask].cpu().numpy())
            all_q3_targets.extend(q3_tgt[mask].cpu().numpy())
    
    # Calculate F1 scores
    f1_q8 = f1_score(all_q8_targets, all_q8_preds, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_targets, all_q3_preds, average='macro', zero_division=0)
    f1_harmonic = harmonic_mean(f1_q8, f1_q3)
    
    avg_loss = total_loss / len(loader)
    
    return avg_loss, f1_q8, f1_q3, f1_harmonic

## 8. Main Training Loop

In [10]:
# # Training history
# history = {
#     'train_loss': [],
#     'train_f1_q8': [],
#     'train_f1_q3': [],
#     'train_f1_harmonic': [],
#     'val_loss': [],
#     'val_f1_q8': [],
#     'val_f1_q3': [],
#     'val_f1_harmonic': [],
# }

# best_val_f1 = 0

# print("Starting training...")
# print(f"Target: F1 Harmonic Mean ≥ 0.60\n")

# for epoch in range(cfg.epochs):
#     # Train
#     train_loss, train_f1_q8, train_f1_q3, train_f1_harm = train_epoch(
#         model, train_loader, optimizer, criterion, cfg.device, epoch
#     )
    
#     # Validate
#     val_loss, val_f1_q8, val_f1_q3, val_f1_harm = validate(
#         model, val_loader, criterion, cfg.device
#     )
    
#     # Update learning rate
#     scheduler.step(val_f1_harm)
    
#     # Save history
#     history['train_loss'].append(train_loss)
#     history['train_f1_q8'].append(train_f1_q8)
#     history['train_f1_q3'].append(train_f1_q3)
#     history['train_f1_harmonic'].append(train_f1_harm)
#     history['val_loss'].append(val_loss)
#     history['val_f1_q8'].append(val_f1_q8)
#     history['val_f1_q3'].append(val_f1_q3)
#     history['val_f1_harmonic'].append(val_f1_harm)
    
#     # Print epoch summary
#     print(f"\nEpoch {epoch+1}/{cfg.epochs} Summary:")
#     print(f"  Train Loss: {train_loss:.4f} | F1 Q8: {train_f1_q8:.4f} | F1 Q3: {train_f1_q3:.4f} | Harmonic: {train_f1_harm:.4f}")
#     print(f"  Val   Loss: {val_loss:.4f} | F1 Q8: {val_f1_q8:.4f} | F1 Q3: {val_f1_q3:.4f} | Harmonic: {val_f1_harm:.4f}")
    
#     # Save checkpoint
#     checkpoint(
#         val_f1_harm,
#         model,
#         optimizer,
#         epoch,
#         {
#             'train_loss': train_loss,
#             'train_f1_harmonic': train_f1_harm,
#             'val_loss': val_loss,
#             'val_f1_q8': val_f1_q8,
#             'val_f1_q3': val_f1_q3,
#             'val_f1_harmonic': val_f1_harm
#         }
#     )
    
#     # Track best validation F1
#     if val_f1_harm > best_val_f1:
#         best_val_f1 = val_f1_harm
    
#     # Log epoch metrics to TrackIO
#     if trk:
#         trk.log({
#             'epoch': epoch + 1,
#             'train/loss_epoch': train_loss,
#             'train/f1_q8_epoch': train_f1_q8,
#             'train/f1_q3_epoch': train_f1_q3,
#             'train/f1_harmonic_epoch': train_f1_harm,
#             'val/loss': val_loss,
#             'val/f1_q8': val_f1_q8,
#             'val/f1_q3': val_f1_q3,
#             'val/f1_harmonic': val_f1_harm,
#             'learning_rate': optimizer.param_groups[0]['lr']
#         })
    
#     print("-" * 80)

# print(f"\n✓ Training complete!")
# print(f"Best Validation F1 Harmonic: {best_val_f1:.4f}")
# print(f"Target achieved: {'✓ YES' if best_val_f1 >= 0.60 else '✗ NO (keep training or tune hyperparameters)'}")

## 9. Plot Training Curves

In [12]:
# fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# # Loss curves
# axes[0].plot(history['train_loss'], label='Train Loss')
# axes[0].plot(history['val_loss'], label='Val Loss')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('Loss')
# axes[0].set_title('Training and Validation Loss')
# axes[0].legend()
# axes[0].grid(True)

# # F1 score curves
# axes[1].plot(history['train_f1_harmonic'], label='Train F1 Harmonic')
# axes[1].plot(history['val_f1_harmonic'], label='Val F1 Harmonic')
# axes[1].plot(history['val_f1_q8'], label='Val F1 Q8', linestyle='--', alpha=0.7)
# axes[1].plot(history['val_f1_q3'], label='Val F1 Q3', linestyle='--', alpha=0.7)
# axes[1].axhline(y=0.60, color='r', linestyle=':', label='Target (0.60)')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('F1 Score')
# axes[1].set_title('F1 Scores Over Epochs')
# axes[1].legend()
# axes[1].grid(True)

# plt.tight_layout()
# plt.savefig('training_curves_bigru.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"Training curves saved to: training_curves_bigru.png")

## 10. Load Best Model and Generate Predictions

In [13]:
# Load best checkpoint
print(f"Loading best model from: {cfg.model_save_path}")
checkpoint_data = torch.load(cfg.model_save_path, map_location=cfg.device)

model.load_state_dict(checkpoint_data['model_state_dict'])
print(f"✓ Loaded model from epoch {checkpoint_data['epoch']+1}")
print(f"  Best Validation F1 Harmonic: {checkpoint_data['best_score']:.4f}")
print(f"  Metrics: {checkpoint_data['metrics']}")

Loading best model from: best_model_bigru.pth
✓ Loaded model from epoch 6
  Best Validation F1 Harmonic: 0.4575
  Metrics: {'train_loss': 1.651529344540198, 'train_f1_harmonic': 0.48589845471041815, 'val_loss': 1.766161968310674, 'val_f1_q8': 0.3373029933760365, 'val_f1_q3': 0.7106933720586931, 'val_f1_harmonic': 0.45748059759437476}


In [14]:
# Load test data
df_test = pd.read_csv(cfg.test_path)
test_ds = ProteinDataset(df_test, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Test samples: {len(test_ds)}")

Test samples: 1816


In [15]:
# Generate predictions
model.eval()
results = []

print("Generating predictions...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        seq = batch['seq'].to(cfg.device)
        ids = batch['ids']
        lengths = batch['lengths']
        
        # Forward pass
        q8_logits, q3_logits = model(seq)
        
        # Get predictions
        q8_preds = q8_logits.argmax(dim=-1).cpu().numpy()
        q3_preds = q3_logits.argmax(dim=-1).cpu().numpy()
        
        # Decode predictions
        for i, id_ in enumerate(ids):
            length = lengths[i].item()
            
            # Extract valid predictions (exclude padding)
            pred_q8_ids = q8_preds[i, :length]
            pred_q3_ids = q3_preds[i, :length]
            
            # Convert to strings
            pred_q8_str = ''.join([ID_TO_Q8[idx] for idx in pred_q8_ids])
            pred_q3_str = ''.join([ID_TO_Q3[idx] for idx in pred_q3_ids])
            
            results.append({
                'id': id_,
                'sst8': pred_q8_str,
                'sst3': pred_q3_str
            })

# Create submission
df_submission = pd.DataFrame(results)
df_submission.to_csv(cfg.submission_path, index=False)

print(f"\n✓ Submission saved to: {cfg.submission_path}")
print(f"  Total predictions: {len(df_submission)}")
print(f"\nFirst 5 predictions:")
print(df_submission.head())

Generating predictions...


Inference: 100%|██████████| 29/29 [00:30<00:00,  1.06s/it]


✓ Submission saved to: submission_bigru.csv
  Total predictions: 1816

First 5 predictions:
   id                                               sst8  \
0   0  CCCCTHHHHHHHHHHHHHHHHHHCSEEEEEETTCTTCCEEEEEECT...   
1   1  CCCCCCCCCCEEEEEEECSTTCEEEEEECTTHHHHHHHHCCCTTHH...   
2   2  CCHHHHHHHHHHHHHHHHHCTHHHHHHHHHHHCCCCCTHHHHHHHH...   
3   3  CCCTTHHHHHHHHHHHHHHHHHHHHHHHCCEEEETTHHHHHHHHHH...   
4   4  CCEEEEESCCCTHHHHHHHHHHHHHHHCEEEEECTTTEEEEEESEE...   

                                                sst3  
0  CCCCCHHHHHHHHHHHHHHHHHHCCEEEEEECCCCCCEEEEEEECC...  
1  CCCCCCCCCCEEEEEEECCCCCEEEEEECCCHHHHHHHCCCCCCHH...  
2  CCHHHHHHHHHHHHHHHHHCCHHHHHHHHHHHCCCCCCCHHHHHHH...  
3  CCCCCHHHHHHHHHHHHHHHHHHHHHHHCCEEEECCHHHHHHHHHH...  
4  CCEEEEECCCCCHHHHHHHHHHHHHHHCCEEEECCCCEEEEECCEE...  


## 11. Submission Validation

In [16]:
# Validate submission format
print("Validating submission...")

# Check all test IDs present
assert len(df_submission) == len(df_test), "Missing test IDs in submission"
assert set(df_submission['id']) == set(df_test['id']), "ID mismatch"

# Check valid characters
valid_q8_chars = set('BCEGIHST')
valid_q3_chars = set('CEH')

for idx, row in df_submission.iterrows():
    # Check Q8
    assert all(c in valid_q8_chars for c in row['sst8']), f"Invalid Q8 character in row {idx}"
    # Check Q3
    assert all(c in valid_q3_chars for c in row['sst3']), f"Invalid Q3 character in row {idx}"
    # Check lengths match
    seq_len = len(df_test.loc[df_test['id'] == row['id'], 'seq'].values[0])
    assert len(row['sst8']) == seq_len, f"Q8 length mismatch in row {idx}"
    assert len(row['sst3']) == seq_len, f"Q3 length mismatch in row {idx}"

print("✓ Submission validation passed!")
print(f"  All {len(df_submission)} predictions are valid")
print(f"  Ready to submit to Kaggle")

Validating submission...
✓ Submission validation passed!
  All 1816 predictions are valid
  Ready to submit to Kaggle


## Summary

### Model Architecture
- **Embedding**: 128 dimensions
- **Bi-GRU**: 3 stacked layers with 512 hidden units per direction
- **Dropout**: 0.3 between layers and before classification heads
- **Total Parameters**: ~20M

### Training Strategy
- **Optimizer**: Adam with weight decay (1e-5)
- **Learning Rate**: 0.001 with ReduceLROnPlateau scheduler
- **Gradient Clipping**: 1.0
- **Batch Size**: 64
- **Epochs**: 30

### Results
- Best Validation F1 Harmonic Mean: *[See training output]*
- Q8 F1 Score: *[See training output]*
- Q3 F1 Score: *[See training output]*

### Next Steps (if target not met)
1. Train for more epochs
2. Try even larger hidden_dim (768, 1024)
3. Implement weighted loss for imbalanced classes
4. Add auxiliary tasks (e.g., solvent accessibility)
5. Ensemble multiple models